In [1]:
%cd ..

c:\Users\HP\OneDrive - University of Moratuwa\Desktop\E-Vision-Projects\DB_SQL_GEN


In [2]:
"""
clarifier.py — ClarifierAgent: detects missing constraints and asks the user.
 
Position in the graph
─────────────────────
  question
     │
     ▼
  clarifier_node          ← THIS FILE
     │
     ├── clarification_needed=True  → return clarification_question to user
     │                                 (graph ends here for this turn)
     │
     └── clarification_needed=False → planner_node → schema → generator → critic
 
Design principles
─────────────────
1. Ask ONCE — gather ALL missing info in a single, friendly message.
2. Infer what can be inferred (time period = current month, etc.).
3. Only block when truly ambiguous — err on the side of proceeding.
4. Fast path: RepCode regex check before calling the LLM (saves tokens).
5. Enriched question merges original + answer so downstream agents
   always receive a complete, unambiguous question.
 
What triggers clarification
────────────────────────────
• RepCode missing AND question is rep-specific (sales, targets, calls, etc.)
• Customer-specific query but no customer name or code given
• Ambiguous time period that cannot be defaulted (e.g. "last quarter" when
  the user means fiscal vs calendar)
• Comparative query missing one of the comparison sides
 
What does NOT trigger clarification (inferred automatically)
─────────────────────────────────────────────────────────────
• Time period → defaults to current month
• Target type → defaults to rep-level (Type=1)
• Result limit → defaults to LIMIT 100
"""

'\nclarifier.py — ClarifierAgent: detects missing constraints and asks the user.\n\nPosition in the graph\n─────────────────────\n  question\n     │\n     ▼\n  clarifier_node          ← THIS FILE\n     │\n     ├── clarification_needed=True  → return clarification_question to user\n     │                                 (graph ends here for this turn)\n     │\n     └── clarification_needed=False → planner_node → schema → generator → critic\n\nDesign principles\n─────────────────\n1. Ask ONCE — gather ALL missing info in a single, friendly message.\n2. Infer what can be inferred (time period = current month, etc.).\n3. Only block when truly ambiguous — err on the side of proceeding.\n4. Fast path: RepCode regex check before calling the LLM (saves tokens).\n5. Enriched question merges original + answer so downstream agents\n   always receive a complete, unambiguous question.\n\nWhat triggers clarification\n────────────────────────────\n• RepCode missing AND question is rep-specific (sales

In [19]:
import re
from typing import Optional, Tuple
 
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from loguru import logger
 
# from src.core.state import AgentState
from src.config import settings

In [20]:
"""
state.py — Extended AgentState with clarification fields.

Drop-in replacement for core/state.py.
Adds 5 new fields for the clarification loop — all other fields unchanged.
"""

from typing import TypedDict, Annotated, List, Optional, Dict, Any
from langchain_core.messages import BaseMessage
import operator


class AgentState(TypedDict):
    """
    State object that flows through the DRGC pipeline.

    New fields vs original
    ──────────────────────
    clarification_needed   : True  → graph routes to clarification response
                             False → graph routes to planner_node
    clarification_question : The question text to show the user
    clarification_answer   : User's answer (fed back in next turn)
    enriched_question      : Original question + clarification merged.
                             The planner always receives this, never the raw question.
    missing_constraints    : List of constraint names that were missing
                             (for logging / debugging)
    """

    # ── Input ─────────────────────────────────────────────────────────────
    question: str

    # ── Clarification Phase (NEW) ─────────────────────────────────────────
    clarification_needed:   Optional[bool]
    clarification_question: Optional[str]
    clarification_answer:   Optional[str]
    enriched_question:      Optional[str]
    missing_constraints:    Optional[List[str]]

    # ── Planning Phase ────────────────────────────────────────────────────
    plan:       Optional[str]
    plan_steps: Optional[List[str]]

    # ── Schema Retrieval Phase ────────────────────────────────────────────
    relevant_tables:  Optional[List[str]]
    schema_context:   Optional[str]
    schema_metadata:  Optional[Dict[str, Any]]

    # ── Generation Phase ──────────────────────────────────────────────────
    sql_query:        Optional[str]
    sql_explanation:  Optional[str]
    few_shot_examples: Optional[List[Dict[str, str]]]

    # ── Execution Phase ───────────────────────────────────────────────────
    query_result:      Optional[Any]
    result_preview:    Optional[str]
    execution_time_ms: Optional[float]

    # ── Error Handling ────────────────────────────────────────────────────
    error:      Optional[str]
    error_type: Optional[str]

    # ── Control Flow ──────────────────────────────────────────────────────
    iterations:   int
    should_retry: bool

    # ── Conversation History ──────────────────────────────────────────────
    messages: Annotated[List[BaseMessage], operator.add]

    # ── Metadata ──────────────────────────────────────────────────────────
    start_time: Optional[float]
    cache_hit:  Optional[bool]

In [4]:
# ── Fast-path: regex patterns (no LLM call needed) ───────────────────────────
 
# Patterns that indicate a RepCode is already present in the question
_REP_CODE_PATTERNS = [
    r'\b[A-Z]{2,6}REP\d{3,6}\b',          # MATREP001, BATREP017
    r'\brep(?:resentative)?\s+[A-Z]{3,}',  # rep MATREP001
    r'\bI\s+am\s+\w+',                      # I am MATREP001
    r'\bmy\s+rep(?:code)?\s+is\b',          # my repcode is
    r'\brepcode[:\s]+\w+',                  # repcode: MATREP001
]
 
# Question types that almost certainly need a RepCode
_REP_REQUIRED_KEYWORDS = [
    'my sales', 'my net sales', 'my target', 'my achievement', 'my productive',
    'my bill', 'my return', 'my outlet', 'my customer', 'my route', 'my net',
    'my sku', 'my eco', 'my volume', 'my coverage',
    'i am rep', 'for rep', 'for the rep',
]
 
# Question types that work without a RepCode (aggregate / all-rep queries)
_NO_REP_KEYWORDS = [
    'all rep', 'all reps', 'all representatives',
    'compare rep', 'each rep', 'by rep', 'per rep',
    'overall', 'company wide', 'company-wide', 'total across',
    'which rep', 'who has', 'top rep', 'best rep',
    'new customer', 'new outlet',     # customer master queries
    'potential outlet',                # outlet master queries
]

In [5]:
def _extract_rep_code(question: str) -> Optional[str]:
    """Return the RepCode if one is explicitly mentioned, else None."""
    for pattern in _REP_CODE_PATTERNS[:1]:   # only the strict code pattern
        m = re.search(pattern, question, re.IGNORECASE)
        if m:
            # Extract just the code token
            code = re.search(r'[A-Z]{2,6}REP\d{3,6}', question, re.IGNORECASE)
            if code:
                return code.group().upper()
    return None

In [6]:
def _needs_rep_code_fast(question: str) -> Tuple[bool, str]:
    """
    Fast regex check: does this question need a RepCode?
 
    Returns (needs_clarification, reason).
    Returns (False, "") if rep-specific but code already present,
    or if query does not require a specific rep.
    """
    q_lower = question.lower()
 
    # If a RepCode is already present → no clarification needed
    for pattern in _REP_CODE_PATTERNS:
        if re.search(pattern, question, re.IGNORECASE):
            return False, ""
 
    # If question is clearly an all-rep or non-rep query → no clarification
    if any(kw in q_lower for kw in _NO_REP_KEYWORDS):
        return False, ""
 
    # If question contains rep-specific "my" keywords → RepCode needed
    if any(kw in q_lower for kw in _REP_REQUIRED_KEYWORDS):
        return True, "RepCode"
 
    return False, ""    # ambiguous — let LLM decide

In [13]:
question = "what is my productive call, I am BATREP001"

In [14]:
_extract_rep_code(question)

'BATREP001'

In [15]:
_needs_rep_code_fast(question)

(False, '')

In [ ]:
# ── LLM-based clarification check ────────────────────────────────────────────
 
_CLARIFIER_PROMPT = """You are a constraint validator for a sales analytics SQL agent.
 
Your job: decide if the user's question has enough information to generate
a correct SQL query, or if critical information is missing.
 
SALES DATABASE CONTEXT
──────────────────────
Tables: sales_flat, sales_targets, sales_hierarchy_nodes, external_parties,
        products, planned_routes, route_customer_assignments.
 
WHAT CAN BE INFERRED (never ask about these)
────────────────────────────────────────────
• Time period → default is current month if not mentioned.
• Target type → default is rep-level (Type=1).
• Result limit → default is top 100 rows.
• Sort order  → default is descending for rankings.
 
WHAT CANNOT BE INFERRED (ask if missing)
─────────────────────────────────────────
• RepCode / Rep name → required for any rep-specific query
  (my sales, my target, my calls, my outlets, etc.).
  If the user says "I am MATREP001" or provides any rep identifier, it IS present.
• CustomerCode / Customer name → required for customer-specific queries.
• ProductCode / Product name → required for product-specific queries
  (unless a category filter is sufficient).
• Comparison period → required only when user says "compare to last month"
  or "vs previous quarter" without specifying the period.
 
RESPONSE FORMAT (strict JSON, no other text)
────────────────────────────────────────────
{{
  "is_sufficient": true | false,
  "missing_constraints": ["RepCode", "CustomerCode", ...],
  "clarification_question": "friendly question to ask the user, or null if sufficient",
  "can_proceed_with_defaults": true | false
}}
 
Rules for clarification_question:
- Ask ALL missing info in ONE friendly message.
- Be specific about what is needed and why.
- Suggest the format (e.g. "your rep code like MATREP001").
- If is_sufficient is true, set clarification_question to null.
 
User Question: {question}"""

In [34]:
class ClarifierAgent:
    """
    Checks if a question has all required constraints before planning.
 
    Fast path: regex check for RepCode (< 0.1ms, no LLM call).
    LLM path:  used only when regex is inconclusive (~300-500ms).
    """
 
    def __init__(self):
        self.llm = ChatAnthropic(
            model   = settings.anthropic_model_fast,
            api_key = settings.anthropic_api_key,
        )
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", _CLARIFIER_PROMPT),
            ("user",   "Validate this question: {question}"),
        ])
        self.chain = self.prompt | self.llm
        
    def check(self, state: AgentState) -> dict:
        """
        Check whether the question is sufficient to answer.
 
        Returns state updates — one of:
          a) clarification_needed=False → proceed to planner
          b) clarification_needed=True  → return question to user
        """
        # Use enriched_question if available (second turn after clarification)
        question = state.get("enriched_question") or state["question"]
        logger.info(f"CLARIFIER: Checking question: {question[:80]}")
 
        # ── Fast path: regex check ────────────────────────────────────────
        needs_rep, reason = _needs_rep_code_fast(question)
 
        if needs_rep:
            clarification_q = (
                "To answer your question I need to know which sales representative "
                "you are referring to. Could you please provide the rep code "
                "(e.g. MATREP001) or rep name?"
            )
            logger.info("CLARIFIER: Fast path → RepCode missing")
            return {
                "clarification_needed":   True,
                "clarification_question": clarification_q,
                "missing_constraints":    ["RepCode"],
                "enriched_question":      question,
            }
 
        # ── LLM path: deeper analysis ─────────────────────────────────────
        try:
            import json
 
            response = self.chain.invoke({"question": question})
            raw      = response.content.strip()
            print(raw)
 
            # Strip markdown fences if present
            raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.DOTALL).strip()
            result   = json.loads(raw)
 
            is_sufficient   = result.get("is_sufficient", True)
            missing         = result.get("missing_constraints", [])
            clarification_q = result.get("clarification_question")
            can_default     = result.get("can_proceed_with_defaults", True)
 
            if not is_sufficient and not can_default and clarification_q:
                logger.info(f"CLARIFIER: LLM path → missing: {missing}")
                return {
                    "clarification_needed":   True,
                    "clarification_question": clarification_q,
                    "missing_constraints":    missing,
                    "enriched_question":      question,
                }
            else:
                logger.info("CLARIFIER: Question is sufficient → proceed")
                return {
                    "clarification_needed":   False,
                    "clarification_question": None,
                    "missing_constraints":    [],
                    "enriched_question":      question,
                }
 
        except Exception as e:
            # On any error, proceed rather than blocking the user
            logger.warning(f"CLARIFIER: LLM check failed ({e}) — proceeding anyway")
            return {
                "clarification_needed": False,
                "enriched_question":    question,
            }
 
    def merge_clarification(self, state: AgentState) -> dict:
        """
        Called when the user has answered the clarification question.
        Merges the original question + clarification answer into enriched_question.
 
        The graph calls this node when clarification_answer is present in state.
        """
        original  = state["question"]
        answer    = state.get("clarification_answer", "").strip()
 
        if not answer:
            return {}
 
        # Reconstruct a complete question from original + answer
        enriched = f"{original} [{answer}]"
        logger.info(f"CLARIFIER: Merged question: {enriched}")
 
        return {
            "enriched_question":    enriched,
            "clarification_answer": None,  # clear so it's not re-merged next turn
        }

In [35]:
# ── LangGraph node functions ──────────────────────────────────────────────────
 
def clarifier_node(state: AgentState) -> dict:
    """First node in the graph. Checks if the question is sufficient."""
    agent = ClarifierAgent()
    return agent.check(state)
 
 
def merge_clarification_node(state: AgentState) -> dict:
    """
    Called when the user has provided a clarification answer.
    Merges it into enriched_question before re-running clarifier_node.
    """
    agent = ClarifierAgent()
    return agent.merge_clarification(state)
 
 
# ── Graph routing function ────────────────────────────────────────────────────
 
def route_after_clarifier(state: AgentState) -> str:
    """
    Conditional edge function for LangGraph.
 
    Returns the name of the next node to route to:
      "needs_clarification" → graph ends, clarification_question returned to user
      "planner"             → proceed with full pipeline
    """
    if state.get("clarification_needed"):
        return "needs_clarification"
    return "planner"

In [36]:
state = AgentState(question="what is rep wise k10 kpi")

planner_node_output = clarifier_node(state)

2026-05-04 16:37:03.967 | INFO     | __main__:check:30 - CLARIFIER: Checking question: what is rep wise k10 kpi
2026-05-04 16:37:07.582 | WARNING  | __main__:check:85 - CLARIFIER: LLM check failed (Extra data: line 7 column 1 (char 128)) — proceeding anyway


```json
{
  "is_sufficient": true,
  "missing_constraints": [],
  "clarification_question": null,
  "can_proceed_with_defaults": true
}
```

**Reasoning:** This is an aggregate/all-rep query asking for K10 KPI broken down by rep ("rep wise"). No specific rep identifier is needed because the user wants to see data across all reps. Time period will default to current month, and results will default to top 100 rows sorted descending.


In [33]:
planner_node_output

{'clarification_needed': False,
 'enriched_question': 'what is rep wise k10 kpi'}